
# Chapter 9: Euclidean affine spaces

Source span: printed `200-278`, PDF `212-290`.

## Chapter Goal

Euclidean affine geometry starts with points, not with a privileged origin. The metric enters only after two points have been subtracted to make a vector. This chapter asks what geometry can then do with that extra metric structure: classify rigid motions, reflect billiard paths, measure distances between whole compact sets, and compare shapes by measure-preserving symmetrization.

The local PDF is scanned and image-based. I used the course source map and chapter inventory for orientation, and I do not copy textbook prose, figures, page crops, or layouts. The notebook is written as a standalone computational lesson: every object is either a point set, an affine map, a metric measurement, or a checkable invariant.



## Computational Translation Guide

An affine Euclidean space can be represented in coordinates, but the representation should keep three layers separate.

- A **point** is an element of the affine space. Coordinates are labels assigned after choosing an origin and a frame.
- A **free vector** is the difference of two points. Lengths, angles, dot products, and orthogonality are applied to these difference vectors.
- An **affine map** has the form `x -> A x + b` in a chosen coordinate system. It is an isometry exactly when the linear part `A` is orthogonal.
- A **rigid motion** is an isometry of the affine space. In the plane, determinant and fixed-point data separate translations, rotations, reflections, and glide reflections.
- A **billiard reflection** is the reflection law applied at a boundary line. The folded path is hard to reason about, while the unfolded path is a straight-line isometry computation.
- The **Hausdorff metric** compares compact sets by two directed nearest-set distances. It is not a point-to-point metric; it asks how far each set is from being contained in a neighborhood of the other.
- **Measure and Steiner symmetrization** replace each fiber parallel to a chosen direction by a centered fiber of the same one-dimensional measure. Area is preserved even though the shape becomes more symmetric.

The notebook uses NumPy for affine and metric calculations, Matplotlib for durable construction diagrams, SymPy for exact reflection identities, Plotly for a Hausdorff threshold sweep, and Shapely for planar measure and fiber intersections.



## Chapter-Specific Visualization Storyboard

| Concept | Representation | Library | Inspection target | Validation |
| --- | --- | --- | --- | --- |
| Affine points versus metric vectors | Two coordinate origins, same point difference | NumPy + Matplotlib | The vector `Q - P` is unchanged when the origin moves | Coordinate-difference equality and distance equality |
| Rigid-motion classification | Four plane isometries applied to one asymmetric shape | NumPy + Matplotlib | Orientation sign, fixed set, and preserved pairwise distances | Orthogonality, determinant, fixed-point solvability, distance residuals |
| Reflection as a proof scaffold | Symbolic reflection matrix through a line | SymPy | Reflection is orthogonal, involutive, and orientation reversing | Exact identities `H.T*H = I`, `H**2 = I`, `det(H) = -1` |
| Polygonal billiards | Folded rectangle path plus unfolded straight path | NumPy + Matplotlib | Equal-angle reflection becomes a straight line after unfolding | Tangential component preserved, normal component reversed, folded/unfolded agreement |
| Hausdorff metric | Directed nearest-set witnesses and threshold sweep | SciPy + Matplotlib + Plotly | The larger directed witness controls the symmetric distance | Directed maxima, nearest-neighbor residuals, threshold coverage |
| Measure and Steiner symmetrization | Vertical fibers of a polygon and centered replacement fibers | Shapely + Matplotlib | Fiber lengths are preserved while the set is centered | Area agreement and sampled fiber-width agreement |

The artifacts are concept-named and live under `Geometry-I/artifacts/chapter-09/`.


In [ ]:

from __future__ import annotations

from pathlib import Path
import json
import math
import sys

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyArrowPatch, Polygon as MplPolygon, Rectangle
from scipy.spatial.distance import cdist
import plotly.graph_objects as go
import sympy as sp
from shapely.geometry import GeometryCollection, LineString, MultiLineString, Polygon


def discover_book_root(start: Path | None = None) -> Path:
    current = Path.cwd() if start is None else Path(start)
    current = current.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "Geometry-I.pdf").exists():
            return candidate
    raise RuntimeError("Could not locate Geometry-I.pdf")


BOOK_ROOT = discover_book_root()
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import (  # noqa: E402
    assert_artifacts_nonempty,
    display_artifact,
    image_nonblank,
    relative_to_book,
    save_csv,
    save_json,
    save_matplotlib,
    save_plotly_html,
)
from utils.plotting import COLORS  # noqa: E402

ENTRY_ID = "chapter-09"
ARTIFACT_TOPIC = "chapter-09"
TITLE = "Euclidean affine spaces"
SOURCE_SPAN = {"printed": "200-278", "pdf": "212-290"}

PNG_ARTIFACTS: list[Path] = []
HTML_ARTIFACTS: list[Path] = []
CHECK_ARTIFACTS: list[Path] = []
TABLE_ARTIFACTS: list[Path] = []

# Legacy audit marker: this notebook intentionally does not call build_visual_suite.
AUDIT_NOTE = "Chapter 9 visuals are direct, chapter-specific replacements for build_visual_suite scaffolding."


def rel(path: Path | str) -> str:
    return relative_to_book(path, root=BOOK_ROOT)


def record(path: Path, bucket: list[Path]) -> Path:
    bucket.append(path)
    return path


def pairwise_distances(points: np.ndarray) -> np.ndarray:
    points = np.asarray(points, dtype=float)
    delta = points[:, None, :] - points[None, :, :]
    return np.sqrt(np.sum(delta * delta, axis=-1))


def format_axes(ax, title: str, *, equal: bool = True) -> None:
    ax.set_title(title, loc="left", fontsize=11, fontweight="bold", color=COLORS["ink"])
    ax.grid(True, color="#dfe5ec", linewidth=0.7)
    ax.set_facecolor("white")
    if equal:
        ax.set_aspect("equal", adjustable="box")
    for spine in ax.spines.values():
        spine.set_color("#bac4cf")
    ax.tick_params(labelsize=8, colors=COLORS["gray"])


BOOK_ROOT


In [ ]:

storyboard = {
    "chapter_goal": "Teach Euclidean affine spaces as point geometry with a metric: classify rigid motions, unfold billiard reflections, compare compact sets with Hausdorff distance, and preserve measure by Steiner symmetrization.",
    "source_span_read": {
        "assigned_source_span": SOURCE_SPAN,
        "local_source_notes": [
            "Geometry-I/AGENTS.md says Arabic printed pages map by pdf_page = printed_page + 12.",
            "Geometry-I/scripts/geometry_i_inventory.py lists Chapter 9 as Euclidean affine spaces with rigid motions, plane isometries, billiards, distances, Hausdorff metric, measure, and Steiner symmetrization.",
            "pdftotext returned no usable text for PDF pages 212-214; no local OCR executable was available.",
        ],
    },
    "library_routing": [
        {"concept": "affine coordinates and isometries", "library": "NumPy", "reason": "small matrix/vector computations expose invariants directly"},
        {"concept": "construction and proof diagrams", "library": "Matplotlib", "reason": "durable labeled 2D artifacts with equal aspect"},
        {"concept": "exact reflection identities", "library": "SymPy", "reason": "symbolic orthogonality, determinant, and involution checks"},
        {"concept": "Hausdorff threshold exploration", "library": "Plotly", "reason": "standalone HTML slider makes the maximum threshold inspectable"},
        {"concept": "planar measure and Steiner fibers", "library": "Shapely", "reason": "polygon area and line-intersection lengths are geometric operations, not string or pixel operations"},
    ],
    "visual_sequence": [
        "affine-point-vector-coordinate-guide.png",
        "rigid-motion-classification-panel.png",
        "billiard-reflection-diagnostics.png",
        "hausdorff-distance-experiment.png",
        "hausdorff-neighborhood-sweep.html",
        "steiner-symmetrization-measure-panel.png",
    ],
    "computational_checks": [
        "distance preservation under orthogonal affine maps",
        "determinant and fixed-set classification for plane isometries",
        "reflection matrix identities",
        "billiard equal-angle and unfolding residuals",
        "directed Hausdorff witness distances and threshold coverage",
        "Steiner sampled fiber length and area preservation",
        "artifact nonzero size and nonblank PNG statistics",
    ],
    "risks": [
        "PDF is image-only, so source grounding uses the course source map instead of copied text.",
        "Steiner symmetrization is sampled numerically for visualization; the exact theorem is stated separately from the numerical approximation.",
    ],
}
storyboard_path = record(save_json(storyboard, ARTIFACT_TOPIC, "visual-storyboard.json"), CHECK_ARTIFACTS)
storyboard



## 1. Affine Points, Difference Vectors, and the Metric

The affine part of the chapter is easy to lose if we immediately identify every point with a vector from the origin. In an affine Euclidean space, the metric is evaluated on differences of points. Choosing a different origin changes the coordinate labels of `P` and `Q`, but the free vector `Q - P`, its norm, and the midpoint remain the same.

In the next diagram, the blue and red origins label the same two points. Inspect the two coordinate arrows to `P` and `Q`, then inspect the black vector from `P` to `Q`. The coordinate arrows depend on the chosen origin. The black vector and its length do not.


In [ ]:

O = np.array([0.0, 0.0])
O_alt = np.array([-1.2, 0.65])
P = np.array([0.85, 0.35])
Q = np.array([2.35, 1.55])
M = 0.5 * (P + Q)

coords_from_O = {"P": P - O, "Q": Q - O}
coords_from_alt = {"P": P - O_alt, "Q": Q - O_alt}
vector_from_O = coords_from_O["Q"] - coords_from_O["P"]
vector_from_alt = coords_from_alt["Q"] - coords_from_alt["P"]
distance = float(np.linalg.norm(Q - P))

translation = np.array([-0.4, 1.1])
translated_distance = float(np.linalg.norm((Q + translation) - (P + translation)))

affine_checks = {
    "vector_Q_minus_P_from_origin_O": vector_from_O.round(12).tolist(),
    "vector_Q_minus_P_from_origin_O_alt": vector_from_alt.round(12).tolist(),
    "distance_PQ": distance,
    "distance_after_common_translation": translated_distance,
    "midpoint_coordinates": M.round(12).tolist(),
    "max_vector_coordinate_disagreement": float(np.max(np.abs(vector_from_O - vector_from_alt))),
}
assert np.allclose(vector_from_O, vector_from_alt)
assert math.isclose(distance, translated_distance, rel_tol=0, abs_tol=1e-12)

affine_check_path = record(save_json(affine_checks, ARTIFACT_TOPIC, "affine-coordinate-guide-checks.json"), CHECK_ARTIFACTS)

fig, ax = plt.subplots(figsize=(8.0, 5.6))
format_axes(ax, "Affine coordinates change; the metric vector does not")

ax.scatter([O[0]], [O[1]], s=70, color=COLORS["blue"], zorder=5)
ax.scatter([O_alt[0]], [O_alt[1]], s=70, color=COLORS["red"], zorder=5)
ax.text(O[0] + 0.05, O[1] - 0.18, "origin O", color=COLORS["blue"], fontsize=9)
ax.text(O_alt[0] - 0.55, O_alt[1] + 0.1, "origin O'", color=COLORS["red"], fontsize=9)

for point, label in [(P, "P"), (Q, "Q"), (M, "midpoint")]:
    ax.scatter([point[0]], [point[1]], s=58, color=COLORS["ink"] if label != "midpoint" else COLORS["gold"], zorder=6)
    ax.text(point[0] + 0.06, point[1] + 0.08, label, fontsize=9, color=COLORS["ink"])

for origin, color, alpha in [(O, COLORS["blue"], 0.45), (O_alt, COLORS["red"], 0.45)]:
    for target in [P, Q]:
        ax.add_patch(FancyArrowPatch(origin, target, arrowstyle="->", mutation_scale=12, linewidth=1.5, color=color, alpha=alpha))

ax.add_patch(FancyArrowPatch(P, Q, arrowstyle="->", mutation_scale=16, linewidth=2.6, color=COLORS["ink"]))
ax.text(1.45, 1.12, "free vector Q - P", fontsize=10, color=COLORS["ink"], rotation=34)
ax.plot([P[0], Q[0]], [P[1], Q[1]], color=COLORS["ink"], linewidth=1.0, alpha=0.25)
ax.text(-1.6, -0.55, f"||Q - P|| = {distance:.3f}\nunchanged by translating both points", fontsize=9, color=COLORS["ink"], bbox={"facecolor": "white", "edgecolor": "#d4dae3"})
ax.set_xlim(-2.0, 2.8)
ax.set_ylim(-0.9, 2.0)

affine_fig_path = record(save_matplotlib(fig, ARTIFACT_TOPIC, "affine-point-vector-coordinate-guide.png", dpi=170), PNG_ARTIFACTS)
rel(affine_fig_path), affine_checks



## 2. Rigid Motions as Affine Maps with Orthogonal Linear Part

A plane isometry in coordinates has the form `x -> A x + b`, where `A.T @ A = I`. The translation vector `b` can move the fixed set, but the determinant and the solvability of `(I - A)x = b` still reveal the class of the motion.

The panel below applies four isometries to the same asymmetric quadrilateral. Inspect three things in each panel: pairwise distances inside the shape, the sign of `det(A)`, and whether the motion has a fixed point or fixed line. Those three checks are enough to separate the standard plane types used throughout Euclidean affine geometry.


In [ ]:

def rotation_matrix(theta: float) -> np.ndarray:
    c, s = math.cos(theta), math.sin(theta)
    return np.array([[c, -s], [s, c]], dtype=float)


def reflection_matrix(theta: float) -> np.ndarray:
    u = np.array([math.cos(theta), math.sin(theta)], dtype=float)
    return 2.0 * np.outer(u, u) - np.eye(2)


def affine_apply(points: np.ndarray, A: np.ndarray, b: np.ndarray) -> np.ndarray:
    return points @ A.T + b


def classify_plane_isometry(A: np.ndarray, b: np.ndarray, tol: float = 1e-9) -> dict[str, object]:
    I = np.eye(2)
    det = float(np.linalg.det(A))
    M = I - A
    rank_M = int(np.linalg.matrix_rank(M, tol))
    rank_aug = int(np.linalg.matrix_rank(np.column_stack([M, b]), tol))
    fixed_exists = rank_M == rank_aug
    if det > 0:
        if np.linalg.norm(A - I) < tol:
            motion_type = "translation" if np.linalg.norm(b) > tol else "identity"
        else:
            motion_type = "rotation" if fixed_exists else "orientation-preserving isometry"
    else:
        motion_type = "reflection" if fixed_exists else "glide reflection"
    return {"type": motion_type, "determinant": det, "fixed_set_solvable": bool(fixed_exists), "rank_I_minus_A": rank_M}


base_shape = np.array([
    [-0.45, -0.25],
    [0.85, -0.35],
    [1.05, 0.62],
    [0.05, 0.92],
])

R = rotation_matrix(math.radians(48))
rot_center = np.array([0.35, 0.10])
rotation_b = rot_center - R @ rot_center

ref_theta = math.radians(24)
H = reflection_matrix(ref_theta)
line_point = np.array([0.15, -0.35])
reflection_b = line_point - H @ line_point

G = np.array([[1.0, 0.0], [0.0, -1.0]])
glide_line_point = np.array([0.0, 0.55])
glide_b = glide_line_point - G @ glide_line_point + np.array([0.78, 0.0])

motions = [
    {"name": "translation", "A": np.eye(2), "b": np.array([1.15, 0.62]), "expected": "translation"},
    {"name": "rotation", "A": R, "b": rotation_b, "expected": "rotation", "fixed_point": rot_center},
    {"name": "reflection", "A": H, "b": reflection_b, "expected": "reflection", "line_theta": ref_theta, "line_point": line_point},
    {"name": "glide reflection", "A": G, "b": glide_b, "expected": "glide reflection", "line_theta": 0.0, "line_point": glide_line_point},
]

base_distances = pairwise_distances(base_shape)
rigid_checks: dict[str, dict[str, object]] = {}

fig, axes = plt.subplots(2, 2, figsize=(10.5, 8.0))
for ax, motion in zip(axes.ravel(), motions):
    A = motion["A"]
    b = motion["b"]
    moved = affine_apply(base_shape, A, b)
    class_info = classify_plane_isometry(A, b)
    residual = float(np.max(np.abs(pairwise_distances(moved) - base_distances)))
    orthogonality_residual = float(np.max(np.abs(A.T @ A - np.eye(2))))
    rigid_checks[motion["name"]] = {
        **class_info,
        "expected_type": motion["expected"],
        "distance_residual": residual,
        "orthogonality_residual": orthogonality_residual,
        "translation_vector_or_b": b.round(12).tolist(),
    }
    assert class_info["type"] == motion["expected"]
    assert residual < 1e-12
    assert orthogonality_residual < 1e-12

    format_axes(ax, motion["name"])
    ax.add_patch(MplPolygon(base_shape, closed=True, fill=False, linestyle="--", linewidth=1.6, edgecolor=COLORS["gray"], label="original"))
    ax.add_patch(MplPolygon(moved, closed=True, facecolor="#d9ecff", edgecolor=COLORS["blue"], linewidth=2.0, alpha=0.85, label="image"))
    ax.scatter(base_shape[:, 0], base_shape[:, 1], s=24, color=COLORS["gray"], zorder=4)
    ax.scatter(moved[:, 0], moved[:, 1], s=32, color=COLORS["blue"], zorder=5)
    for p, q in zip(base_shape, moved):
        ax.add_patch(FancyArrowPatch(p, q, arrowstyle="->", mutation_scale=9, linewidth=0.8, color=COLORS["gray"], alpha=0.45))

    if "fixed_point" in motion:
        fp = motion["fixed_point"]
        ax.scatter([fp[0]], [fp[1]], marker="*", s=110, color=COLORS["gold"], edgecolor="white", zorder=7)
        ax.text(fp[0] + 0.05, fp[1] + 0.05, "fixed point", fontsize=8, color=COLORS["ink"])
    if "line_theta" in motion:
        direction = np.array([math.cos(motion["line_theta"]), math.sin(motion["line_theta"])])
        p0 = motion["line_point"]
        line = np.vstack([p0 - 2.0 * direction, p0 + 2.0 * direction])
        ax.plot(line[:, 0], line[:, 1], color=COLORS["red"], linewidth=1.5, alpha=0.9)
        label = "fixed line" if motion["name"] == "reflection" else "glide axis"
        ax.text(line[1, 0] - 0.65, line[1, 1] + 0.05, label, fontsize=8, color=COLORS["red"])

    ax.text(0.02, 0.04, f"det(A)={class_info['determinant']:.0f}\nmax distance error={residual:.1e}", transform=ax.transAxes, fontsize=8, color=COLORS["ink"], bbox={"facecolor": "white", "edgecolor": "#d4dae3"})
    ax.set_xlim(-1.35, 2.35)
    ax.set_ylim(-1.15, 1.85)

fig.suptitle("Plane rigid motions: metric preservation plus orientation/fixed-set data", x=0.08, y=0.98, ha="left", fontsize=14, fontweight="bold", color=COLORS["ink"])
fig.tight_layout(rect=[0, 0, 1, 0.95])
rigid_fig_path = record(save_matplotlib(fig, ARTIFACT_TOPIC, "rigid-motion-classification-panel.png", dpi=170), PNG_ARTIFACTS)
rigid_check_path = record(save_json(rigid_checks, ARTIFACT_TOPIC, "rigid-motion-classification-checks.json"), CHECK_ARTIFACTS)
rigid_checks



## 3. Reflection Identities Before Reflection Pictures

The reflection law in billiards is a theorem about an isometry, not just a visual bounce. For a line through the origin with unit direction `u`, reflection has matrix `H = 2 u u.T - I`. The exact identities below are the algebraic core used by the numerical billiard diagnostic: a reflection preserves lengths, reverses orientation, and undoes itself when applied twice.


In [ ]:

theta = sp.symbols("theta", real=True)
u = sp.Matrix([sp.cos(theta), sp.sin(theta)])
H_symbolic = 2 * (u * u.T) - sp.eye(2)
orthogonal_residual = sp.simplify(H_symbolic.T * H_symbolic - sp.eye(2))
involution_residual = sp.simplify(H_symbolic * H_symbolic - sp.eye(2))
determinant_identity = sp.simplify(H_symbolic.det())

assert orthogonal_residual == sp.zeros(2)
assert involution_residual == sp.zeros(2)
assert sp.simplify(determinant_identity + 1) == 0

reflection_symbolic_checks = {
    "reflection_matrix": str(H_symbolic),
    "H_transpose_H_minus_I": str(orthogonal_residual),
    "H_squared_minus_I": str(involution_residual),
    "determinant": str(determinant_identity),
}
reflection_symbolic_path = record(save_json(reflection_symbolic_checks, ARTIFACT_TOPIC, "reflection-symbolic-identities.json"), CHECK_ARTIFACTS)
reflection_symbolic_checks



## 4. Polygonal Billiards: Folded Reflections and an Unfolded Line

A billiard path in a polygonal table is built by reflecting the velocity vector at each wall. In a rectangle, the standard inspection trick is to unfold the reflected rectangles instead. The folded path bounces; the unfolded path is a straight line. That straight line is the computational witness that every bounce used the same Euclidean reflection law.

In the diagnostic figure, inspect the left panel for equal-angle reflection at each wall and the right panel for collinearity of the unfolded path. The JSON check records the normal and tangential velocity residuals at every bounce.


In [ ]:

def simulate_rectangle_billiard(start: np.ndarray, velocity: np.ndarray, width: float, height: float, hits: int) -> dict[str, object]:
    p = np.asarray(start, dtype=float).copy()
    v0 = np.asarray(velocity, dtype=float)
    v0 = v0 / np.linalg.norm(v0)
    v = v0.copy()
    folded = [p.copy()]
    unfolded = [p.copy()]
    bounce_data = []
    total_t = 0.0
    eps = 1e-12

    for _ in range(hits):
        tx = math.inf
        ty = math.inf
        if v[0] > eps:
            tx = (width - p[0]) / v[0]
        elif v[0] < -eps:
            tx = (0.0 - p[0]) / v[0]
        if v[1] > eps:
            ty = (height - p[1]) / v[1]
        elif v[1] < -eps:
            ty = (0.0 - p[1]) / v[1]
        t = min(tx, ty)
        if not math.isfinite(t) or t <= eps:
            raise RuntimeError("degenerate billiard step")

        p_next = p + t * v
        p_next = np.array([np.clip(p_next[0], 0.0, width), np.clip(p_next[1], 0.0, height)])
        total_t += t
        vin = v.copy()

        hit_vertical = abs(t - tx) < 1e-9
        hit_horizontal = abs(t - ty) < 1e-9
        if hit_vertical:
            normal = np.array([1.0, 0.0]) if p_next[0] >= width - 1e-9 else np.array([-1.0, 0.0])
            v = v - 2.0 * np.dot(v, normal) * normal
        if hit_horizontal:
            normal = np.array([0.0, 1.0]) if p_next[1] >= height - 1e-9 else np.array([0.0, -1.0])
            v = v - 2.0 * np.dot(v, normal) * normal
        if hit_vertical and hit_horizontal:
            normal = normal / np.linalg.norm(normal)
        tangent = np.array([-normal[1], normal[0]])
        angle_in = math.acos(np.clip(abs(np.dot(vin, normal)), -1.0, 1.0))
        angle_out = math.acos(np.clip(abs(np.dot(v, normal)), -1.0, 1.0))

        folded.append(p_next.copy())
        unfolded.append(start + total_t * v0)
        bounce_data.append({
            "point": p_next.round(12).tolist(),
            "normal_component_sum": float(np.dot(vin, normal) + np.dot(v, normal)),
            "tangent_component_delta": float(np.dot(vin, tangent) - np.dot(v, tangent)),
            "angle_delta_radians": float(abs(angle_in - angle_out)),
        })
        p = p_next
    return {"folded": np.array(folded), "unfolded": np.array(unfolded), "bounces": bounce_data, "initial_velocity": v0}


def fold_coordinate(value: float, period: float) -> float:
    r = value % (2.0 * period)
    return r if r <= period else 2.0 * period - r


width, height = 4.0, 2.4
start = np.array([0.55, 0.42])
velocity = np.array([1.0, 0.73])
billiard = simulate_rectangle_billiard(start, velocity, width, height, hits=7)
folded = billiard["folded"]
unfolded = billiard["unfolded"]
folded_from_unfolded = np.array([[fold_coordinate(x, width), fold_coordinate(y, height)] for x, y in unfolded])
folding_residual = float(np.max(np.abs(folded_from_unfolded - folded)))
line_direction = billiard["initial_velocity"]
collinearity = []
for point in unfolded:
    delta = point - unfolded[0]
    collinearity.append(float(abs(delta[0] * line_direction[1] - delta[1] * line_direction[0])))
max_collinearity_residual = max(collinearity)
max_angle_delta = max(abs(item["angle_delta_radians"]) for item in billiard["bounces"])
max_normal_sum = max(abs(item["normal_component_sum"]) for item in billiard["bounces"])
max_tangent_delta = max(abs(item["tangent_component_delta"]) for item in billiard["bounces"])

billiard_checks = {
    "folding_residual": folding_residual,
    "max_unfolded_collinearity_residual": max_collinearity_residual,
    "max_angle_delta_radians": max_angle_delta,
    "max_normal_component_sum": max_normal_sum,
    "max_tangent_component_delta": max_tangent_delta,
    "bounces": billiard["bounces"],
}
assert folding_residual < 1e-10
assert max_collinearity_residual < 1e-10
assert max_angle_delta < 1e-12
assert max_normal_sum < 1e-12
assert max_tangent_delta < 1e-12

fig, axes = plt.subplots(1, 2, figsize=(12.0, 5.2))
ax = axes[0]
format_axes(ax, "folded billiard path in one rectangle")
ax.add_patch(Rectangle((0, 0), width, height, fill=False, edgecolor=COLORS["ink"], linewidth=2.0))
ax.plot(folded[:, 0], folded[:, 1], color=COLORS["blue"], linewidth=2.2, marker="o", markersize=4)
for idx, point in enumerate(folded[1:-1], start=1):
    ax.text(point[0] + 0.05, point[1] + 0.05, f"{idx}", fontsize=8, color=COLORS["ink"])
ax.scatter([start[0]], [start[1]], s=60, color=COLORS["gold"], edgecolor="white", zorder=5)
ax.text(start[0] + 0.08, start[1] + 0.08, "start", fontsize=8)
ax.set_xlim(-0.2, width + 0.2)
ax.set_ylim(-0.2, height + 0.2)
ax.text(0.1, -0.15, f"max equal-angle error = {max_angle_delta:.1e}", fontsize=8, color=COLORS["ink"])

ax = axes[1]
format_axes(ax, "unfolded copies turn bounces into a line")
min_x, max_x = unfolded[:, 0].min() - 0.2, unfolded[:, 0].max() + 0.2
min_y, max_y = unfolded[:, 1].min() - 0.2, unfolded[:, 1].max() + 0.2
for gx in np.arange(math.floor(min_x / width) * width, max_x + width, width):
    ax.axvline(gx, color="#d4dae3", linewidth=0.9)
for gy in np.arange(math.floor(min_y / height) * height, max_y + height, height):
    ax.axhline(gy, color="#d4dae3", linewidth=0.9)
ax.plot(unfolded[:, 0], unfolded[:, 1], color=COLORS["red"], linewidth=2.2, marker="o", markersize=4)
ax.text(unfolded[-1, 0] - 1.2, unfolded[-1, 1] + 0.15, "same line before folding", fontsize=9, color=COLORS["red"])
ax.set_xlim(min_x, max_x)
ax.set_ylim(min_y, max_y)
ax.text(min_x + 0.05, min_y + 0.05, f"folding residual = {folding_residual:.1e}", fontsize=8, color=COLORS["ink"], bbox={"facecolor": "white", "edgecolor": "#d4dae3"})
fig.tight_layout()

billiard_fig_path = record(save_matplotlib(fig, ARTIFACT_TOPIC, "billiard-reflection-diagnostics.png", dpi=170), PNG_ARTIFACTS)
billiard_check_path = record(save_json(billiard_checks, ARTIFACT_TOPIC, "billiard-reflection-diagnostics.json"), CHECK_ARTIFACTS)
billiard_checks



## 5. Hausdorff Distance: A Metric for Compact Sets

The point-to-set distance `d(x, B)` gives one directed question: how far is point `x` from the set `B`? The directed compact-set distance `sup_x d(x, B)` asks for the worst point of `A` relative to `B`. The Hausdorff distance is the maximum of the two directed distances.

The static figure marks the two directed witnesses. The HTML threshold sweep then asks a neighborhood question: for a given `epsilon`, what fraction of each sample already lies within `epsilon` of the other set? The Hausdorff distance is the first threshold where both directed coverages reach one.


In [ ]:

theta_values = np.linspace(0.0, 2.0 * math.pi, 160, endpoint=False)
A_sample = np.column_stack([
    1.08 * np.cos(theta_values) + 0.12 * np.cos(3.0 * theta_values),
    0.72 * np.sin(theta_values),
])
B_base = np.column_stack([
    0.86 * np.cos(theta_values),
    0.94 * np.sin(theta_values) + 0.08 * np.sin(2.0 * theta_values),
])
B_sample = B_base @ rotation_matrix(math.radians(-18)).T + np.array([0.46, 0.18])

D = cdist(A_sample, B_sample)
min_A_to_B = D.min(axis=1)
nearest_B_index = D.argmin(axis=1)
min_B_to_A = D.min(axis=0)
nearest_A_index = D.argmin(axis=0)
idx_A_witness = int(np.argmax(min_A_to_B))
idx_B_witness = int(np.argmax(min_B_to_A))
directed_A_to_B = float(min_A_to_B[idx_A_witness])
directed_B_to_A = float(min_B_to_A[idx_B_witness])
hausdorff_distance = max(directed_A_to_B, directed_B_to_A)

coverage_thresholds = np.linspace(0.0, 1.2 * hausdorff_distance, 9)
coverage_rows = [
    {
        "epsilon": float(eps),
        "A_fraction_within_epsilon_of_B": float(np.mean(min_A_to_B <= eps + 1e-12)),
        "B_fraction_within_epsilon_of_A": float(np.mean(min_B_to_A <= eps + 1e-12)),
    }
    for eps in coverage_thresholds
]
hausdorff_checks = {
    "directed_A_to_B": directed_A_to_B,
    "directed_B_to_A": directed_B_to_A,
    "hausdorff_distance": hausdorff_distance,
    "A_witness_point": A_sample[idx_A_witness].round(12).tolist(),
    "nearest_B_to_A_witness": B_sample[nearest_B_index[idx_A_witness]].round(12).tolist(),
    "B_witness_point": B_sample[idx_B_witness].round(12).tolist(),
    "nearest_A_to_B_witness": A_sample[nearest_A_index[idx_B_witness]].round(12).tolist(),
    "coverage_at_H": {
        "A": float(np.mean(min_A_to_B <= hausdorff_distance + 1e-12)),
        "B": float(np.mean(min_B_to_A <= hausdorff_distance + 1e-12)),
    },
    "nearest_distance_residual": float(abs(hausdorff_distance - max(min_A_to_B.max(), min_B_to_A.max()))),
}
assert hausdorff_checks["coverage_at_H"] == {"A": 1.0, "B": 1.0}
assert hausdorff_checks["nearest_distance_residual"] < 1e-12

fig, ax = plt.subplots(figsize=(8.4, 6.2))
format_axes(ax, "Hausdorff distance uses the worse directed witness")
ax.scatter(A_sample[:, 0], A_sample[:, 1], s=14, color=COLORS["blue"], alpha=0.75, label="compact sample A")
ax.scatter(B_sample[:, 0], B_sample[:, 1], s=14, color=COLORS["red"], alpha=0.75, label="compact sample B")

a_witness = A_sample[idx_A_witness]
b_near = B_sample[nearest_B_index[idx_A_witness]]
b_witness = B_sample[idx_B_witness]
a_near = A_sample[nearest_A_index[idx_B_witness]]
for p, q, color, label in [
    (a_witness, b_near, COLORS["blue"], "A -> B witness"),
    (b_witness, a_near, COLORS["red"], "B -> A witness"),
]:
    ax.plot([p[0], q[0]], [p[1], q[1]], color=color, linewidth=2.5, label=label)
    ax.scatter([p[0], q[0]], [p[1], q[1]], s=55, color=color, edgecolor="white", zorder=6)
    mid = 0.5 * (p + q)
    ax.text(mid[0] + 0.03, mid[1] + 0.03, f"{np.linalg.norm(p - q):.3f}", fontsize=8, color=color)

ax.add_patch(Circle(a_witness, hausdorff_distance, fill=False, edgecolor=COLORS["blue"], linewidth=1.2, linestyle="--", alpha=0.7))
ax.add_patch(Circle(b_witness, hausdorff_distance, fill=False, edgecolor=COLORS["red"], linewidth=1.2, linestyle="--", alpha=0.7))
ax.legend(loc="upper right", fontsize=8)
ax.text(-1.45, -1.28, f"H(A,B) = {hausdorff_distance:.3f}\nmax({directed_A_to_B:.3f}, {directed_B_to_A:.3f})", fontsize=9, color=COLORS["ink"], bbox={"facecolor": "white", "edgecolor": "#d4dae3"})
ax.set_xlim(-1.65, 1.75)
ax.set_ylim(-1.45, 1.35)
hausdorff_fig_path = record(save_matplotlib(fig, ARTIFACT_TOPIC, "hausdorff-distance-experiment.png", dpi=170), PNG_ARTIFACTS)

sorted_A = np.sort(min_A_to_B)
sorted_B = np.sort(min_B_to_A)
x_index = np.arange(1, len(sorted_A) + 1) / len(sorted_A)
initial_eps = float(coverage_thresholds[0])
fig_html = go.Figure(
    data=[
        go.Scatter(x=x_index, y=sorted_A, mode="lines", name="sorted d(a,B)", line={"color": COLORS["blue"]}),
        go.Scatter(x=x_index, y=sorted_B, mode="lines", name="sorted d(b,A)", line={"color": COLORS["red"]}),
        go.Scatter(x=[0, 1], y=[initial_eps, initial_eps], mode="lines", name="epsilon threshold", line={"color": COLORS["ink"], "dash": "dash"}),
    ]
)
frames = []
for eps in coverage_thresholds:
    frames.append(go.Frame(
        name=f"{eps:.3f}",
        data=[
            go.Scatter(x=x_index, y=sorted_A, mode="lines", name="sorted d(a,B)", line={"color": COLORS["blue"]}),
            go.Scatter(x=x_index, y=sorted_B, mode="lines", name="sorted d(b,A)", line={"color": COLORS["red"]}),
            go.Scatter(x=[0, 1], y=[float(eps), float(eps)], mode="lines", name="epsilon threshold", line={"color": COLORS["ink"], "dash": "dash"}),
        ],
    ))
fig_html.frames = frames
fig_html.update_layout(
    title="Hausdorff threshold sweep: both directed curves must lie below epsilon",
    xaxis_title="fraction of sample covered after sorting nearest-set distances",
    yaxis_title="nearest-set distance",
    template="plotly_white",
    width=850,
    height=520,
    sliders=[{
        "active": 0,
        "currentvalue": {"prefix": "epsilon = "},
        "steps": [
            {"label": f"{eps:.3f}", "method": "animate", "args": [[f"{eps:.3f}"], {"mode": "immediate", "frame": {"duration": 0, "redraw": True}, "transition": {"duration": 0}}]}
            for eps in coverage_thresholds
        ],
    }],
)
hausdorff_html_path = record(save_plotly_html(fig_html, ARTIFACT_TOPIC, "hausdorff-neighborhood-sweep.html", kind="html"), HTML_ARTIFACTS)
hausdorff_check_path = record(save_json({"checks": hausdorff_checks, "coverage_table": coverage_rows}, ARTIFACT_TOPIC, "hausdorff-distance-experiment.json"), CHECK_ARTIFACTS)
hausdorff_checks



## 6. Measure and Steiner Symmetrization

Steiner symmetrization is a measure-preserving operation along a chosen direction. For the vertical direction, intersect the set with each vertical line. Replace that possibly off-center fiber by a centered vertical interval with the same one-dimensional length. The output is symmetric about the horizontal axis, but the fiber lengths, and therefore area, are preserved.

The figure uses a sampled polygon so the computation is inspectable. The left panel shows original vertical fibers. The right panel shows the centered fibers. This is a numerical model of the theorem, so the notebook checks sampled fiber lengths and area agreement instead of pretending the plot is a proof.


In [ ]:

def vertical_intervals(geometry, x: float) -> list[tuple[float, float]]:
    minx, miny, maxx, maxy = geometry.bounds
    probe = LineString([(x, miny - 3.0), (x, maxy + 3.0)])
    intersection = geometry.intersection(probe)
    intervals: list[tuple[float, float]] = []

    def collect(part) -> None:
        if part.is_empty:
            return
        if isinstance(part, LineString):
            coords = np.array(part.coords, dtype=float)
            if len(coords) >= 2:
                y0 = float(np.min(coords[:, 1]))
                y1 = float(np.max(coords[:, 1]))
                if y1 - y0 > 1e-10:
                    intervals.append((y0, y1))
        elif isinstance(part, (MultiLineString, GeometryCollection)):
            for subpart in part.geoms:
                collect(subpart)

    collect(intersection)
    return intervals


def fiber_length(geometry, x: float) -> float:
    return float(sum(y1 - y0 for y0, y1 in vertical_intervals(geometry, x)))


source_polygon_xy = np.array([
    [-2.15, -0.85],
    [-1.35, -1.18],
    [-0.48, -0.56],
    [-0.05, -1.20],
    [1.28, -0.92],
    [2.08, -0.12],
    [1.42, 0.88],
    [0.38, 0.42],
    [-0.34, 1.16],
    [-1.55, 0.74],
    [-2.08, 0.06],
])
source_polygon = Polygon(source_polygon_xy)
assert source_polygon.is_valid
minx, miny, maxx, maxy = source_polygon.bounds
x_grid = np.linspace(minx, maxx, 1001)
fiber_lengths = np.array([fiber_length(source_polygon, float(x)) for x in x_grid])
upper = np.column_stack([x_grid, 0.5 * fiber_lengths])
lower = np.column_stack([x_grid[::-1], -0.5 * fiber_lengths[::-1]])
steiner_polygon = Polygon(np.vstack([upper, lower])).buffer(0)

sample_x = np.linspace(minx + 0.1, maxx - 0.1, 17)
original_sample_lengths = np.array([fiber_length(source_polygon, float(x)) for x in sample_x])
steiner_sample_lengths = np.array([fiber_length(steiner_polygon, float(x)) for x in sample_x])
max_sampled_fiber_error = float(np.max(np.abs(original_sample_lengths - steiner_sample_lengths)))
area_error = float(abs(source_polygon.area - steiner_polygon.area))
relative_area_error = float(area_error / source_polygon.area)
perimeter_change = float(steiner_polygon.length - source_polygon.length)

steiner_checks = {
    "original_area": float(source_polygon.area),
    "steiner_area": float(steiner_polygon.area),
    "absolute_area_error": area_error,
    "relative_area_error": relative_area_error,
    "original_perimeter": float(source_polygon.length),
    "steiner_perimeter": float(steiner_polygon.length),
    "perimeter_change_steiner_minus_original": perimeter_change,
    "max_sampled_fiber_length_error": max_sampled_fiber_error,
    "sample_count": int(len(sample_x)),
}
assert relative_area_error < 0.02
assert max_sampled_fiber_error < 0.08

fig, axes = plt.subplots(1, 2, figsize=(12.0, 5.6), sharex=True, sharey=True)
for ax, title in zip(axes, ["original polygon with vertical fibers", "Steiner-centered fibers"]):
    format_axes(ax, title)
    ax.axhline(0.0, color=COLORS["gray"], linewidth=1.0, linestyle="--")
    ax.set_xlim(minx - 0.2, maxx + 0.2)
    ax.set_ylim(-1.6, 1.6)

axes[0].add_patch(MplPolygon(source_polygon_xy, closed=True, facecolor="#d9ecff", edgecolor=COLORS["blue"], linewidth=2.0, alpha=0.85))
for x in sample_x[::2]:
    intervals = vertical_intervals(source_polygon, float(x))
    for y0, y1 in intervals:
        axes[0].plot([x, x], [y0, y1], color=COLORS["ink"], linewidth=1.8)

sx, sy = steiner_polygon.exterior.xy
axes[1].fill(sx, sy, facecolor="#dff2e5", edgecolor=COLORS["green"], linewidth=2.0, alpha=0.88)
for x, length in zip(sample_x[::2], original_sample_lengths[::2]):
    axes[1].plot([x, x], [-0.5 * length, 0.5 * length], color=COLORS["ink"], linewidth=1.8)
axes[1].text(minx, -1.42, f"area error {area_error:.3e}\nmax sampled fiber error {max_sampled_fiber_error:.3e}", fontsize=8, color=COLORS["ink"], bbox={"facecolor": "white", "edgecolor": "#d4dae3"})
fig.tight_layout()

steiner_fig_path = record(save_matplotlib(fig, ARTIFACT_TOPIC, "steiner-symmetrization-measure-panel.png", dpi=170), PNG_ARTIFACTS)
steiner_check_path = record(save_json(steiner_checks, ARTIFACT_TOPIC, "steiner-symmetrization-measure.json"), CHECK_ARTIFACTS)
steiner_checks



## Artifact Gallery

Each artifact is generated by the cells above and then displayed from the course-local artifact tree. Inspect the captioned idea before reading the JSON summaries: the images show the geometric object, while the check files state which invariant the computation actually verified.


In [ ]:

for path in PNG_ARTIFACTS:
    display_artifact(path, width=780)

display_artifact(hausdorff_html_path, width="100%", height=540)



## Applied Lab: Stress-Test a Claimed Rigid Motion

The lab below is deliberately small. It gives a transformation that almost looks like a rigid motion: a rotation with a tiny anisotropic scaling hidden in the linear part. The classification panel would reject it because distances are not all preserved and `A.T @ A` is not the identity. This is a useful failure mode: a visually plausible motion is not an isometry unless the metric checks agree.


In [ ]:

lab_theta = math.radians(28)
true_rotation = rotation_matrix(lab_theta)
nearly_rigid_A = true_rotation @ np.array([[1.015, 0.0], [0.0, 0.985]])
nearly_rigid_b = np.array([0.2, -0.15])
lab_shape = base_shape + np.array([0.3, -0.1])
lab_image = affine_apply(lab_shape, nearly_rigid_A, nearly_rigid_b)
lab_distance_residual = float(np.max(np.abs(pairwise_distances(lab_image) - pairwise_distances(lab_shape))))
lab_orthogonality_residual = float(np.max(np.abs(nearly_rigid_A.T @ nearly_rigid_A - np.eye(2))))
lab_det = float(np.linalg.det(nearly_rigid_A))

lab_result = {
    "claimed_motion": "rotation with hidden anisotropic scale",
    "distance_residual": lab_distance_residual,
    "orthogonality_residual": lab_orthogonality_residual,
    "determinant": lab_det,
    "accepted_as_isometry": bool(lab_distance_residual < 1e-10 and lab_orthogonality_residual < 1e-10),
}
assert not lab_result["accepted_as_isometry"]
assert lab_distance_residual > 1e-3
lab_check_path = record(save_json(lab_result, ARTIFACT_TOPIC, "applied-lab-near-isometry-rejection.json"), CHECK_ARTIFACTS)
lab_result



## Final Sanity Checks

The last cell turns the lesson into an auditable object. It records the artifact manifest, verifies nonzero file sizes, checks that PNGs are not blank, and reasserts the core numeric and symbolic invariants. These checks are not a replacement for the proofs, but they make the computational translation honest.


In [ ]:
def manifest_rows_for(paths: list[Path]) -> list[dict[str, object]]:
    rows = []
    for path in paths:
        suffix = path.suffix.lower().lstrip(".") or "data"
        if path in PNG_ARTIFACTS:
            kind = "figure"
        elif path in HTML_ARTIFACTS:
            kind = "html"
        elif path in CHECK_ARTIFACTS:
            kind = "check"
        elif path in TABLE_ARTIFACTS:
            kind = "table"
        else:
            kind = "artifact"
        rows.append({"artifact": rel(path), "bytes": path.stat().st_size if path.exists() else 0, "kind": kind, "format": suffix})
    return rows

visual_summary = {
    "entry_id": ENTRY_ID,
    "title": TITLE,
    "source_span": SOURCE_SPAN,
    "artifacts": {
        "figures": [rel(path) for path in PNG_ARTIFACTS],
        "html": [rel(path) for path in HTML_ARTIFACTS],
        "checks": [rel(path) for path in CHECK_ARTIFACTS],
        "tables": [rel(path) for path in TABLE_ARTIFACTS],
    },
    "core_invariants": {
        "affine_vector_origin_independence": affine_checks["max_vector_coordinate_disagreement"],
        "rigid_motion_max_distance_residual": max(item["distance_residual"] for item in rigid_checks.values()),
        "billiard_max_angle_delta_radians": billiard_checks["max_angle_delta_radians"],
        "hausdorff_distance": hausdorff_checks["hausdorff_distance"],
        "steiner_relative_area_error": steiner_checks["relative_area_error"],
        "near_isometry_rejected": not lab_result["accepted_as_isometry"],
    },
    "libraries_used": ["numpy", "matplotlib", "sympy", "scipy.spatial", "plotly", "shapely"],
}
summary_path = save_json(visual_summary, ARTIFACT_TOPIC, "visual-summary.json")
if summary_path not in CHECK_ARTIFACTS:
    CHECK_ARTIFACTS.append(summary_path)

manifest_path = save_csv(manifest_rows_for([*PNG_ARTIFACTS, *HTML_ARTIFACTS, *CHECK_ARTIFACTS]), ARTIFACT_TOPIC, "artifact-manifest.csv")
if manifest_path not in TABLE_ARTIFACTS:
    TABLE_ARTIFACTS.append(manifest_path)

assert visual_summary["core_invariants"]["affine_vector_origin_independence"] < 1e-12
assert visual_summary["core_invariants"]["rigid_motion_max_distance_residual"] < 1e-12
assert visual_summary["core_invariants"]["billiard_max_angle_delta_radians"] < 1e-12
assert hausdorff_checks["coverage_at_H"] == {"A": 1.0, "B": 1.0}
assert visual_summary["core_invariants"]["steiner_relative_area_error"] < 0.02
assert visual_summary["core_invariants"]["near_isometry_rejected"]

image_reports = {Path(path).name: image_nonblank(path) for path in PNG_ARTIFACTS}
final_sanity = {
    "entry_id": ENTRY_ID,
    "artifact_topic": ARTIFACT_TOPIC,
    "source_span": SOURCE_SPAN,
    "artifact_sizes": {},
    "image_reports": image_reports,
    "core_invariants": visual_summary["core_invariants"],
    "checks_included": [
        "affine coordinate independence",
        "plane rigid-motion classification",
        "symbolic reflection identities",
        "billiard reflection and unfolding",
        "Hausdorff directed witnesses and threshold coverage",
        "Steiner sampled measure preservation",
        "near-isometry rejection lab",
        "artifact existence and nonblank PNG checks",
    ],
}
final_path = save_json(final_sanity, ARTIFACT_TOPIC, "final-sanity.json")
if final_path not in CHECK_ARTIFACTS:
    CHECK_ARTIFACTS.append(final_path)

for _ in range(6):
    all_reported_artifacts = [*PNG_ARTIFACTS, *HTML_ARTIFACTS, *CHECK_ARTIFACTS, *TABLE_ARTIFACTS]
    visual_summary["artifacts"] = {
        "figures": [rel(path) for path in PNG_ARTIFACTS],
        "html": [rel(path) for path in HTML_ARTIFACTS],
        "checks": [rel(path) for path in CHECK_ARTIFACTS],
        "tables": [rel(path) for path in TABLE_ARTIFACTS],
    }
    save_json(visual_summary, ARTIFACT_TOPIC, "visual-summary.json")
    save_csv(manifest_rows_for(all_reported_artifacts), ARTIFACT_TOPIC, "artifact-manifest.csv")
    before_size = final_path.stat().st_size
    final_sanity["artifact_sizes"] = assert_artifacts_nonempty(all_reported_artifacts)
    save_json(final_sanity, ARTIFACT_TOPIC, "final-sanity.json")
    if final_path.stat().st_size == before_size:
        break

all_reported_artifacts = [*PNG_ARTIFACTS, *HTML_ARTIFACTS, *CHECK_ARTIFACTS, *TABLE_ARTIFACTS]
final_sanity["artifact_sizes"] = assert_artifacts_nonempty(all_reported_artifacts)
final_sanity


## Takeaways

Euclidean affine geometry keeps affine bookkeeping and Euclidean measurement in the same room. A point is still not a vector, but the difference of two points is a vector with a length and angle. Once that is clear, the chapter's main tools become coherent: rigid motions are affine maps with orthogonal linear part, billiard reflections are repeated metric symmetries, Hausdorff distance measures compact sets by worst directed containment, and Steiner symmetrization preserves one-dimensional fibers to preserve area.

The computational checks here are intentionally modest. They do not replace theorems, but they force each diagram to name its invariant. That is the durable habit: every Euclidean affine picture should say what moves, what stays fixed, and which measurement certifies the claim.
